# 在 `netsol/resume-score-details` 上单卡微调 Gemma 4 E4B-it

本 Notebook 仿照情绪分类微调文件的结构，改为使用 Hugging Face 上的 `netsol/resume-score-details` 数据集，对 Gemma 4 E4B-it 做 LoRA 微调，使模型生成“AI时代计算机相关岗位职业匹配模拟器”的结构化报告 JSON。

与原情绪分类 Notebook 不同：

1. **数据集改为 Hugging Face 的 `netsol/resume-score-details`**，因为当前目标数据集不在原 Notebook 使用的 ModelScope 镜像中。
2. **任务从情绪分类改为职业匹配报告生成**，输出不再是单个标签，而是固定字段的 JSON。
3. **评估指标改为生成任务指标**：JSON 合法率、字段完整率、匹配分合法率、有效样本数。
4. **保留单卡、BF16、LoRA、ModelScope 下载 Gemma 模型**，不使用 `bitsandbytes` 4bit，适配 AMD / ROCm 环境。
5. **不加入 RAG、岗位图谱、外部爬取数据**，只做本数据集相关的最小可运行微调。

> 注意：如果你使用 AMD ROCm，PyTorch 中仍然通过 `torch.cuda` 访问 GPU，这是 PyTorch 的统一接口。

## 1. 安装依赖

这里只安装本 Notebook 必需的包：

- `modelscope`：从魔搭下载 Gemma 模型。
- `datasets`：从 Hugging Face 加载 `netsol/resume-score-details`。
- `transformers`：加载 Gemma 模型和 tokenizer。
- `trl`：使用 `SFTTrainer` 做指令微调。
- `peft`：配置 LoRA。
- `pandas` / `tqdm`：保存和查看评估结果。

如果环境已经装好，可以跳过本单元。

In [ ]:
# AMD / ROCm 云环境通常已预装匹配的 torch。这里不使用 -U，避免把 ROCm torch 升级成 CUDA 版。
!uv pip install modelscope transformers accelerate datasets trl peft pandas tqdm sentencepiece protobuf --no-cache -i https://mirrors.cloud.tencent.com/pypi/simple/

## 2. 导入依赖与全局配置

默认使用单卡 BF16 LoRA 微调。显存不够时优先降低：

```python
TRAIN_LIMIT
EVAL_LIMIT
per_device_train_batch_size
max_length
```

如果显卡不支持 BF16，把：

```python
MODEL_DTYPE = torch.float16
BF16 = False
FP16 = True
```

In [ ]:
import os

# 强制使用 HF 镜像 - 必须在 import datasets 之前设置
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import DatasetDict, load_dataset, load_from_disk
from modelscope import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

# 额外强制设置 datasets 库使用镜像
import datasets
datasets.config.HF_ENDPOINT = "https://hf-mirror.com"

warnings.filterwarnings("ignore")

# -----------------------------
# 基础配置
# -----------------------------
# 魔搭上的 Gemma 4 E4B-it 模型 ID。
MODELSCOPE_MODEL_ID = "google/gemma-4-E4B-it"

# Hugging Face 上的简历-JD 匹配评分数据集。
HF_DATASET_ID = "netsol/resume-score-details"
HF_ENDPOINT = os.environ["HF_ENDPOINT"]
LOCAL_DATASET_DIR = "./datasets/resume_score_details"

# 微调输出目录。
OUTPUT_DIR = "./gemma4-it-career-report-lora-single-gpu"

# 数据量控制。先用小数据跑通，确认没问题后再加大或设为 None。
TRAIN_LIMIT = None
VALIDATION_LIMIT = 120
EVAL_LIMIT = 60

SEED = 42
MODEL_DTYPE = torch.bfloat16
BF16 = True
FP16 = False

REQUIRED_REPORT_FIELDS = [
    "valid_profile_and_job",
    "recommended_role",
    "match_score",
    "match_level",
    "match_reasons",
    "ability_profile",
    "gap_analysis",
    "learning_plan",
    "summary",
]

SYSTEM_PROMPT = """你是一个AI时代计算机相关岗位职业匹配模拟器。
你的任务是根据用户简历、岗位描述、岗位最低要求和评分标准，生成结构化职业匹配报告。
必须只输出合法JSON，不要输出Markdown，不要输出额外解释。"""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./datasets", exist_ok=True)

print("torch version:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("HIP version:", getattr(torch.version, "hip", None))
print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))

if "+cu" in torch.__version__ and not torch.cuda.is_available():
    raise RuntimeError("当前环境安装的是 CUDA 版 torch，且 GPU 不可用。AMD/ROCm 云环境请恢复 ROCm 版 torch 后再继续。")
if torch.cuda.is_available() and getattr(torch.version, "hip", None) is None:
    print("Warning: 当前 torch 未显示 HIP 版本。如果这是 AMD 云环境，请确认没有误装 CUDA 版 torch。")

## 3. 固定随机种子

固定随机种子可以让数据 shuffle、LoRA 初始化和训练过程尽量可复现。

In [ ]:
def setup_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

setup_seed(SEED)

## 4. 从魔搭 ModelScope 下载 Gemma 模型

模型仍然沿用原 Notebook 的 ModelScope 下载方式，不需要从 Hugging Face 下载模型权重。

如果是第一次访问 Gemma 系列模型，可能需要先在 ModelScope 页面登录并接受许可协议。

In [ ]:
print("Downloading model from ModelScope...")
print("ModelScope model id:", MODELSCOPE_MODEL_ID)

model_dir = snapshot_download(
    MODELSCOPE_MODEL_ID,
    cache_dir="./models",
)

print("Downloaded model dir:", model_dir)
LOCAL_MODEL_DIR = model_dir

## 5. 加载 `netsol/resume-score-details` 数据集

这里优先从本地离线目录加载数据集；如果本地目录不存在，则通过 Hugging Face 镜像加载公开数据集。与原 Notebook 不同，本数据集不是 ModelScope 的 emotion 镜像，因此不使用 `dataset_snapshot_download()`。

加载后会自动切分出 `train` / `validation`：

- 本地目录默认是 `./datasets/resume_score_details`，可由 `save_to_disk()` 生成。
- 在线加载使用 `HF_ENDPOINT = "https://hf-mirror.com"`。
- 如果数据集本身有多个 split，则优先使用原 split。
- 如果只有 `train`，则按 90% / 10% 切分。

In [ ]:
print("Loading dataset...")
print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))

# 直接从 HuggingFace 镜像加载数据集
print("Loading dataset from Hugging Face mirror...")
from datasets import DownloadConfig

download_config = DownloadConfig(
    resume_download=True,
    max_retries=10,
)

raw_dataset = load_dataset(
    HF_DATASET_ID, 
    cache_dir="./datasets",
    download_config=download_config,
)

# 保存到本地，下次可以直接用 load_from_disk
print(f"Saving dataset to {LOCAL_DATASET_DIR} for future use...")
raw_dataset.save_to_disk(LOCAL_DATASET_DIR)
print("Dataset saved!")

print("Raw dataset:", raw_dataset)
print("Splits:", list(raw_dataset.keys()))

if "train" not in raw_dataset:
    first_split = list(raw_dataset.keys())[0]
    raw_dataset = DatasetDict({"train": raw_dataset[first_split]})

if "validation" not in raw_dataset:
    split_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=SEED)
    raw_dataset = DatasetDict({
        "train": split_dataset["train"],
        "validation": split_dataset["test"],
    })

def maybe_limit(split, limit):
    split = split.shuffle(seed=SEED)
    if limit is None:
        return split
    return split.select(range(min(limit, len(split))))

dataset = DatasetDict({
    "train": maybe_limit(raw_dataset["train"], TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
})

print(dataset)
print("columns:", dataset["train"].column_names)
print("example keys:", dataset["train"][0].keys())
print("example:")
print(dataset["train"][0])

## 6. 构造职业匹配报告 SFT 数据

原数据集是简历-JD 匹配评分数据，不是职业规划报告数据。这里只做必要转换：

- `prompt`：system + user，包含 resume、job_description、minimum_requirements、macro/micro criteria。
- `completion`：assistant，输出固定字段 JSON。

报告字段固定为：

```json
{
  "valid_profile_and_job": true,
  "recommended_role": "...",
  "match_score": 0,
  "match_level": "...",
  "match_reasons": [],
  "ability_profile": {"strengths": [], "weaknesses": []},
  "gap_analysis": [],
  "learning_plan": [],
  "summary": "..."
}
```

In [ ]:
def safe_get(obj, path, default=None):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        return [value]
    return [str(value)]

def normalize_score(sample):
    candidates = []
    aggregated = safe_get(sample, ["output", "scores", "aggregated_scores"], {})
    if isinstance(aggregated, dict):
        candidates.extend([
            aggregated.get("macro_scores"),
            aggregated.get("micro_scores"),
            aggregated.get("overall_score"),
            aggregated.get("total_score"),
        ])
    candidates.extend([
        safe_get(sample, ["output", "score"]),
        safe_get(sample, ["score"]),
    ])

    values = []
    for item in candidates:
        try:
            if item is not None:
                values.append(float(item))
        except Exception:
            pass

    if not values:
        return 50

    avg = sum(values) / len(values)
    if avg <= 5:
        return round(avg / 5 * 100)
    if avg <= 10:
        return round(avg / 10 * 100)
    return max(0, min(100, round(avg)))

def match_level(score):
    if score >= 85:
        return "高度匹配"
    if score >= 70:
        return "较高匹配"
    if score >= 55:
        return "中等匹配"
    if score >= 40:
        return "较低匹配"
    return "不匹配"

def extract_skills(sample):
    details = sample.get("details", {}) if isinstance(sample.get("details"), dict) else {}
    skills = details.get("skills", [])
    skills = [str(x) for x in as_list(skills) if str(x).strip()]
    return skills[:8]

def extract_weaknesses(sample):
    requirements = safe_get(sample, ["output", "scores", "requirements"], [])
    weaknesses = []
    if isinstance(requirements, list):
        for item in requirements:
            if isinstance(item, dict) and item.get("meets") is False:
                text = item.get("criteria") or item.get("requirement") or item.get("description")
                if text:
                    weaknesses.append(str(text))
    return weaknesses[:5]

def build_report(sample):
    output = sample.get("output", {}) if isinstance(sample.get("output"), dict) else {}
    score = normalize_score(sample)
    reasons = as_list(output.get("justification"))[:5]
    skills = extract_skills(sample)
    weaknesses = extract_weaknesses(sample)

    if not reasons:
        reasons = ["根据简历经历与岗位描述，候选人与目标岗位存在一定匹配度。"]
    if not skills:
        skills = ["具备部分岗位相关经历"]
    if not weaknesses:
        weaknesses = ["需要进一步补充目标岗位的关键项目经验"]

    level = match_level(score)
    return {
        "valid_profile_and_job": bool(output.get("valid_resume_and_jd", True)),
        "recommended_role": "根据岗位描述匹配的目标岗位",
        "match_score": score,
        "match_level": level,
        "match_reasons": reasons,
        "ability_profile": {
            "strengths": skills,
            "weaknesses": weaknesses,
        },
        "gap_analysis": weaknesses,
        "learning_plan": [
            "围绕目标岗位补充核心技能项目",
            "将已有经历整理为可量化的项目成果",
            "根据岗位最低要求补齐缺失能力",
            "准备与岗位职责对应的作品集或案例说明",
        ],
        "summary": f"该候选人与目标岗位的匹配度为{score}分，整体属于{level}。建议结合岗位要求补齐短板，并突出已有技能与项目经历。",
    }

def is_usable(sample):
    input_data = sample.get("input", {}) if isinstance(sample.get("input"), dict) else {}
    resume = input_data.get("resume", "")
    jd = input_data.get("job_description", "")
    return isinstance(resume, str) and isinstance(jd, str) and len(resume.strip()) >= 100 and len(jd.strip()) >= 100

def to_prompt_completion(sample):
    input_data = sample.get("input", {}) if isinstance(sample.get("input"), dict) else {}
    user_payload = {
        "resume": input_data.get("resume", ""),
        "job_description": input_data.get("job_description", ""),
        "minimum_requirements": input_data.get("minimum_requirements", []),
        "macro_criteria": input_data.get("macro_dict", {}),
        "micro_criteria": input_data.get("micro_dict", {}),
        "additional_info": input_data.get("additional_info", ""),
    }

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
        "completion": [
            {"role": "assistant", "content": json.dumps(build_report(sample), ensure_ascii=False)},
        ],
    }

filtered_dataset = DatasetDict({
    split: dataset[split].filter(is_usable)
    for split in dataset.keys()
})

sft_dataset = filtered_dataset.map(
    to_prompt_completion,
    remove_columns=filtered_dataset["train"].column_names,
)

print(filtered_dataset)
print(sft_dataset)
print(sft_dataset["train"][0])

## 7. 加载 tokenizer 和基础模型

这里从魔搭下载后的本地路径加载模型。

单卡版本要点：

- 不使用 `device_map="auto"` 做多卡切分。
- 不使用 `bitsandbytes` 4bit。
- 开启 `gradient_checkpointing=True` 节省显存。
- 如果 tokenizer 缺少 chat template，则从同一 ModelScope 仓库读取 `chat_template.jinja` 作为兜底。

In [ ]:
print("Loading tokenizer from:", LOCAL_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_DIR,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

def _load_official_gemma_chat_template() -> str:
    template_dir = snapshot_download(
        MODELSCOPE_MODEL_ID,
        cache_dir="./models",
        allow_file_pattern=["chat_template.jinja"],
    )
    path = os.path.join(template_dir, "chat_template.jinja")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

if not getattr(tokenizer, "chat_template", None):
    print("Loading official chat_template.jinja...")
    tokenizer.chat_template = _load_official_gemma_chat_template()
else:
    print("tokenizer.chat_template already set, leaving as-is.")

_probe = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
print("chat_template probe output:\n" + _probe)

print("Loading base model from:", LOCAL_MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

base_model.to(device)
base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Base model loaded.")
print("Base model device:", next(base_model.parameters()).device)

## 8. 推理辅助函数

微调前后都用同一套推理函数。这里要求模型输出 JSON，并尽量从生成文本中提取第一个 JSON 对象。

In [ ]:
def extract_json_object(text: str):
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except Exception:
        return None

def build_user_payload_from_sample(sample):
    input_data = sample.get("input", {}) if isinstance(sample.get("input"), dict) else {}
    return {
        "resume": input_data.get("resume", ""),
        "job_description": input_data.get("job_description", ""),
        "minimum_requirements": input_data.get("minimum_requirements", []),
        "macro_criteria": input_data.get("macro_dict", {}),
        "micro_criteria": input_data.get("micro_dict", {}),
        "additional_info": input_data.get("additional_info", ""),
    }

def generate_report(model, tokenizer, user_payload, max_new_tokens=700):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
    ]
    device = next(model.parameters()).device
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

sample_payload = build_user_payload_from_sample(filtered_dataset["validation"][0])
print(generate_report(base_model, tokenizer, sample_payload)[:1000])

## 9. 评估函数

本任务是结构化报告生成，不使用分类准确率。这里评估：

- `json_valid_rate`：能否解析为合法 JSON。
- `field_complete_rate`：是否包含全部必需字段。
- `score_valid_rate`：`match_score` 是否在 0-100。
- `evaluated_examples`：实际评估样本数。

In [ ]:
def evaluate_model(model, tokenizer, split="validation", limit=EVAL_LIMIT):
    raw_source = filtered_dataset[split]
    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    rows = []
    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        payload = build_user_payload_from_sample(ex)
        raw_text = generate_report(model, tokenizer, payload)
        parsed = extract_json_object(raw_text)

        json_valid = isinstance(parsed, dict)
        fields_complete = json_valid and all(field in parsed for field in REQUIRED_REPORT_FIELDS)
        score = parsed.get("match_score") if json_valid else None
        score_valid = isinstance(score, (int, float)) and 0 <= score <= 100

        rows.append({
            "json_valid": json_valid,
            "fields_complete": fields_complete,
            "score_valid": score_valid,
            "raw_prediction": raw_text,
            "parsed_prediction": json.dumps(parsed, ensure_ascii=False) if parsed is not None else "",
        })

    pred_df = pd.DataFrame(rows)
    total = len(pred_df)
    metrics = {
        "json_valid_rate": float(pred_df["json_valid"].mean()) if total else 0.0,
        "field_complete_rate": float(pred_df["fields_complete"].mean()) if total else 0.0,
        "score_valid_rate": float(pred_df["score_valid"].mean()) if total else 0.0,
        "evaluated_examples": total,
    }
    return metrics, pred_df

## 10. 微调前评估

先评估基础模型生成职业匹配报告 JSON 的能力，作为 LoRA 微调后的对比基线。

如果只想快速训练，可以把 `RUN_PRE_EVAL = False`。

In [ ]:
RUN_PRE_EVAL = True

if RUN_PRE_EVAL:
    pre_metrics, pre_preds = evaluate_model(base_model, tokenizer, split="validation", limit=EVAL_LIMIT)
else:
    pre_metrics, pre_preds = {}, pd.DataFrame()

pre_metrics

## 11. 配置 LoRA

沿用原 Notebook 的简洁配置：

```python
target_modules="all-linear"
```

这样可以避免手动匹配不同 Gemma 版本内部层名。

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

## 12. 定义训练参数

与原 Notebook 的主要不同：

- `max_length` 从情绪分类的短文本长度提高到 `4096`，因为简历和 JD 更长。
- `per_device_train_batch_size` 降为 `1`，避免长上下文 OOM。
- 仍然使用 `adamw_torch`，避免 AMD ROCm 下 `bitsandbytes` 优化器兼容问题。

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=20,
    num_train_epochs=2,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,

    bf16=BF16,
    fp16=FP16,
    tf32=False,

    max_length=4096,
    packing=False,
    completion_only_loss=True,

    remove_unused_columns=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

## 13. 开始 LoRA 微调

训练前会检查 LoRA 参数是否真的被挂上。如果可训练参数为 0，说明 `target_modules` 没匹配成功，需要调整 LoRA 配置。

In [ ]:
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainable_params = 0
total_params = 0
trainable_param_names = []

for name, param in trainer.model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        trainable_param_names.append(name)

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters were attached. Check target_modules before training.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable ratio: {100 * trainable_params / total_params:.4f}%")
print("Example trainable parameters:")
print(trainable_param_names[:20])

train_result = trainer.train()

trainer.model.eval()
trainer.model.config.use_cache = True
train_result

## 14. 保存 LoRA adapter 和 tokenizer

保存的是 LoRA adapter，不是完整大模型权重。

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print("Saved adapter and tokenizer to:", OUTPUT_DIR)

## 15. 微调后评估

训练后直接复用内存里的模型评估，避免重新加载导致显存碎片或 OOM。

In [ ]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True

post_metrics, post_preds = evaluate_model(ft_model, tokenizer, split="validation", limit=EVAL_LIMIT)
post_metrics

## 16. 对比微调前后效果

这里合并微调前后的结构化生成指标，方便写报告。

In [ ]:
comparison_rows = []
if pre_metrics:
    comparison_rows.append({"stage": "pre_finetuning", **pre_metrics})
comparison_rows.append({"stage": "post_finetuning", **post_metrics})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 17. 手动测试微调后的模型

这里输入一个简历和岗位描述，观察模型是否能稳定输出职业匹配报告 JSON。

In [ ]:
test_payload = {
    "resume": "本科软件工程，熟悉 Python、SQL、机器学习基础，完成过电商用户行为分析项目和 Flask 后端项目，能够使用 pandas 做数据清洗和可视化。",
    "job_description": "招聘数据分析师，负责业务数据分析、SQL取数、指标监控、可视化报表和业务洞察，支持产品和运营决策。",
    "minimum_requirements": [
        "熟悉 SQL",
        "具备 Python 数据分析能力",
        "理解常见业务指标",
        "有数据可视化经验"
    ],
    "macro_criteria": {"technical_skills": 50, "project_experience": 30, "communication": 20},
    "micro_criteria": {"SQL": 30, "Python": 30, "data_visualization": 20, "business_understanding": 20},
    "additional_info": "优先考虑有电商分析项目经验的候选人。"
}

raw_report = generate_report(ft_model, tokenizer, test_payload)
print(raw_report)
print("\nParsed JSON:")
print(json.dumps(extract_json_object(raw_report), ensure_ascii=False, indent=2))

## 18. 保存评估结果

输出文件会保存到 `OUTPUT_DIR` 中，方便后续写报告或对比。

In [ ]:
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "career_report_before_after_metrics.csv"), index=False)
pre_preds.to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_predictions.csv"), index=False)
post_preds.to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_predictions.csv"), index=False)

print("Saved all outputs to:", OUTPUT_DIR)

## 19. 可选：重新加载本地 LoRA adapter 推理

以后不想重新训练，可以用下面代码重新加载：

1. 从魔搭下载的本地基础模型目录加载 base model。
2. 从 `OUTPUT_DIR` 加载 LoRA adapter。
3. 使用同一个 `generate_report()` 做推理。

In [ ]:
RUN_RELOAD_TEST = False

if RUN_RELOAD_TEST:
    reload_tokenizer = AutoTokenizer.from_pretrained(
        OUTPUT_DIR,
        use_fast=True,
        trust_remote_code=True,
    )
    if reload_tokenizer.pad_token is None:
        reload_tokenizer.pad_token = reload_tokenizer.eos_token

    reload_base_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_DIR,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    reload_base_model.to(device)

    reload_model = PeftModel.from_pretrained(
        reload_base_model,
        OUTPUT_DIR,
    )
    reload_model.eval()

    print(generate_report(reload_model, reload_tokenizer, test_payload))

## 20. 常见问题

### 1. 与原情绪分类 Notebook 有哪些关键不同？

- 数据集从 `AI-ModelScope/emotion` 改为 `netsol/resume-score-details`。
- 数据集下载从 ModelScope 改为本地 `load_from_disk()` 优先，缺失时再通过 Hugging Face 镜像 `load_dataset()`。
- 输出从单个情绪标签改为职业匹配报告 JSON。
- 评估从 accuracy / F1 改为 JSON 合法率、字段完整率、分数合法率。
- `max_length` 从短文本分类长度改为 `4096`，batch size 相应降为 `1`。

### 2. Hugging Face 数据集下载失败怎么办？

第 5 节已经设置 `HF_ENDPOINT = "https://hf-mirror.com"`。如果镜像仍不可访问，可以先在本地或其他环境下载数据集并执行 `save_to_disk()`，再上传到 `./datasets/resume_score_details`，notebook 会自动走 `load_from_disk()`。

### 2b. AMD / ROCm 环境里 GPU 不可用怎么办？

不要在安装依赖时升级 `torch`。如果第 2 节显示 `torch.__version__` 包含 `+cu` 且 `torch.cuda.is_available()` 为 `False`，说明 ROCm 版 PyTorch 被 CUDA 版覆盖了，需要恢复云环境自带的 ROCm PyTorch 或重建环境后再运行。

### 3. 显存不够怎么办？

优先调整：

```python
TRAIN_LIMIT = 300
VALIDATION_LIMIT = 50
EVAL_LIMIT = 20
max_length = 2048
gradient_accumulation_steps = 16
```

### 4. 想用全量数据怎么办？

保持：

```python
TRAIN_LIMIT = None
```

`netsol/resume-score-details` 是 1K 级别数据，通常可以全量训练。